# ML Distribuido — Regresión de litros en tanques (Horeb)
### Versión corregida — Sesión 4

Este notebook parte de tu versión original y aplica 4 correcciones que se discutieron tras revisar los resultados:

1. **Split temporal en vez de aleatorio** — con datos de sensores (lecturas cada 10 min) un `randomSplit` mezcla timestamps vecinos entre train y test, lo que infla las métricas. Ahora se entrena con el 80% de fechas más antiguas y se prueba con el 20% más reciente.
2. **`CrossValidator` en vez de un solo split para elegir el modelo** — antes se comparaban configuraciones evaluando una sola vez contra el mismo test set. Ahora se usa validación cruzada (folds) sobre el set de entrenamiento para elegir hiperparámetros, y el test temporal se toca **una sola vez**, al final, con el mejor modelo ya elegido.
3. **Se eliminan `caudal_lpm` y `presion_linea_kpa`** de los predictores — tenían correlación e importancia prácticamente nulas con `litros`.
4. **Nuevo "Modelo B" universal** — el notebook original solo servía para `horeb_master_01` (el único dispositivo con los 5 sensores extendidos). Aquí se agrega un segundo modelo entrenado con **los 8 dispositivos**, usando solo las variables que todos comparten, para que sí sea desplegable en toda la flota.

Cada celda de código trae un comentario por línea (o por bloque) explicando qué hace.


## 1. Sesión de Spark

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("sesion4-ml-distribuido-regresion-tanques-v2")  # nombre que aparece en la Spark UI
    .master("local[*]")                                      # usa todos los cores disponibles de la máquina
    .config("spark.ui.port", "4040")                          # puerto fijo para la Spark UI
    .config("spark.sql.shuffle.partitions", "8")              # particiones al hacer shuffle (joins/groupBy) — bajo porque el dataset es chico
    .config("spark.driver.memory", "4g")                      # memoria del driver
    .getOrCreate()                                            # reutiliza la sesión si ya existe, si no la crea
)
spark.sparkContext.setLogLevel("ERROR")  # baja el ruido de logs de Spark en la salida del notebook
spark

## 2. Rutas del proyecto

In [2]:
import os

# El CSV debe estar en la misma carpeta que este notebook, dentro de data/
RUTA = "data/tanques_horeb_muestra.csv"

ARTIFACTS = "artifacts"          # carpeta donde se guardarán el parquet y los modelos entrenados
os.makedirs(ARTIFACTS, exist_ok=True)   # la crea si no existe; no falla si ya existe
print(f"ARTIFACTS = {os.path.abspath(ARTIFACTS)}")

ARTIFACTS = /home/jovyan/artifacts


## 3. Carga exploratoria (inferSchema)\nIgual que en el notebook original: una primera lectura rápida para ver qué tipos de dato detecta Spark solo.

In [3]:
# Lectura rápida dejando que Spark adivine los tipos de columna (inferSchema=True)
# Esto es solo exploratorio: es más lento porque Spark tiene que leer el archivo dos veces
# (una para inferir tipos, otra para cargar los datos)
df = spark.read.csv(RUTA, header=True, inferSchema=True)
df.printSchema()

root
 |-- device_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- zona: string (nullable = true)
 |-- distancia_cm: double (nullable = true)
 |-- temp_aire_c: double (nullable = true)
 |-- hum_aire_pct: double (nullable = true)
 |-- nivel_cm_comp: double (nullable = true)
 |-- litros: double (nullable = true)
 |-- capacidad_litros: integer (nullable = true)
 |-- altura_tanque_cm: integer (nullable = true)
 |-- presion_linea_kpa: double (nullable = true)
 |-- caudal_lpm: double (nullable = true)
 |-- temp_agua_c: double (nullable = true)
 |-- orp_mv: double (nullable = true)
 |-- horas_residencia: double (nullable = true)
 |-- bateria_pct: double (nullable = true)



## 4. Carga con schema explícito\nEn producción no conviene usar `inferSchema`: es lento y puede adivinar mal un tipo. Se define el schema a mano.

In [4]:
from pyspark.sql.types import (
    StructType, StructField, StringType, TimestampType, DoubleType
)

# Se define explícitamente el tipo de cada columna, en el mismo orden del CSV
schema = StructType([
    StructField("device_id",         StringType(),    nullable=False),  # identificador del tanque/sensor
    StructField("timestamp",         TimestampType(), nullable=False),  # momento de la lectura
    StructField("zona",              StringType(),    nullable=True),   # zona geográfica del tanque
    StructField("distancia_cm",      DoubleType(),    nullable=True),   # distancia del sensor a la superficie del agua
    StructField("temp_aire_c",       DoubleType(),    nullable=True),   # temperatura del aire
    StructField("hum_aire_pct",      DoubleType(),    nullable=True),   # humedad del aire
    StructField("nivel_cm_comp",     DoubleType(),    nullable=True),   # nivel de agua compensado (calculado a partir de distancia_cm)
    StructField("litros",            DoubleType(),    nullable=True),   # variable objetivo: litros de agua en el tanque
    StructField("capacidad_litros",  DoubleType(),    nullable=True),   # capacidad total del tanque (varía por dispositivo)
    StructField("altura_tanque_cm",  DoubleType(),    nullable=True),   # altura física del tanque (varía por dispositivo)
    StructField("presion_linea_kpa", DoubleType(),    nullable=True),   # presión de línea — solo existe en horeb_master_01
    StructField("caudal_lpm",        DoubleType(),    nullable=True),   # caudal — solo existe en horeb_master_01
    StructField("temp_agua_c",       DoubleType(),    nullable=True),   # temperatura del agua — solo existe en horeb_master_01
    StructField("orp_mv",            DoubleType(),    nullable=True),   # potencial redox del agua — solo existe en horeb_master_01
    StructField("horas_residencia",  DoubleType(),    nullable=True),   # horas que el agua lleva en el tanque — solo existe en horeb_master_01
    StructField("bateria_pct",       DoubleType(),    nullable=True),   # batería del sensor
])

df_raw = (
    spark.read
    .option("header", True)                            # el CSV trae encabezado
    .option("timestampFormat", "yyyy-MM-dd HH:mm:ss")   # formato exacto de la columna timestamp
    .schema(schema)                                     # aplica el schema definido arriba (evita inferSchema)
    .csv(RUTA)
)

print(f"Filas: {df_raw.count():,}")   # count() dispara una acción distribuida real (recorre todas las particiones)
df_raw.printSchema()

Filas: 34,560
root
 |-- device_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- zona: string (nullable = true)
 |-- distancia_cm: double (nullable = true)
 |-- temp_aire_c: double (nullable = true)
 |-- hum_aire_pct: double (nullable = true)
 |-- nivel_cm_comp: double (nullable = true)
 |-- litros: double (nullable = true)
 |-- capacidad_litros: double (nullable = true)
 |-- altura_tanque_cm: double (nullable = true)
 |-- presion_linea_kpa: double (nullable = true)
 |-- caudal_lpm: double (nullable = true)
 |-- temp_agua_c: double (nullable = true)
 |-- orp_mv: double (nullable = true)
 |-- horas_residencia: double (nullable = true)
 |-- bateria_pct: double (nullable = true)



## 5. Nulos globales por columna

In [5]:
from pyspark.sql.functions import col, count as spark_count, when

# Para cada columna c, cuenta cuántas filas tienen esa columna en null
df_raw.select([
    spark_count(when(col(c).isNull(), c)).alias(c)   # cuenta solo cuando la columna es null, si no cuenta None (se ignora)
    for c in df_raw.columns
]).show(vertical=True, truncate=False)

-RECORD 0------------------
 device_id         | 0     
 timestamp         | 0     
 zona              | 0     
 distancia_cm      | 0     
 temp_aire_c       | 0     
 hum_aire_pct      | 0     
 nivel_cm_comp     | 207   
 litros            | 207   
 capacidad_litros  | 0     
 altura_tanque_cm  | 0     
 presion_linea_kpa | 30240 
 caudal_lpm        | 30240 
 temp_agua_c       | 30240 
 orp_mv            | 30240 
 horas_residencia  | 30240 
 bateria_pct       | 0     



## 6. Nulos segmentados por `device_id` — el hallazgo clave

Aquí está la razón de todas las decisiones que vienen después: los nulos no están distribuidos al azar,
están concentrados por completo en 7 de los 8 dispositivos.

In [6]:
# Mismo conteo de nulos, pero agrupado por dispositivo, para las 5 columnas de sensores "extendidos"
cols_extendidas = ["presion_linea_kpa", "caudal_lpm", "temp_agua_c", "orp_mv", "horas_residencia"]

df_raw.groupBy("device_id").agg(
    *[spark_count(when(col(c).isNull(), c)).alias(c) for c in cols_extendidas]
).orderBy("device_id").show(truncate=False)

# Conclusión: horeb_master_01 tiene 0 nulos en estas 5 columnas.
# horeb_n02 a horeb_n08 tienen el 100% de sus filas en null: esos sensores no existen en esos dispositivos.

+---------------+-----------------+----------+-----------+------+----------------+
|device_id      |presion_linea_kpa|caudal_lpm|temp_agua_c|orp_mv|horas_residencia|
+---------------+-----------------+----------+-----------+------+----------------+
|horeb_master_01|0                |0         |0          |0     |0               |
|horeb_n02      |4320             |4320      |4320       |4320  |4320            |
|horeb_n03      |4320             |4320      |4320       |4320  |4320            |
|horeb_n04      |4320             |4320      |4320       |4320  |4320            |
|horeb_n05      |4320             |4320      |4320       |4320  |4320            |
|horeb_n06      |4320             |4320      |4320       |4320  |4320            |
|horeb_n07      |4320             |4320      |4320       |4320  |4320            |
|horeb_n08      |4320             |4320      |4320       |4320  |4320            |
+---------------+-----------------+----------+-----------+------+----------------+



## 7. Estadísticas descriptivas

In [7]:
VARIABLES_NUMERICAS = [
    "distancia_cm", "temp_aire_c", "hum_aire_pct", "litros",
    "presion_linea_kpa", "caudal_lpm", "temp_agua_c",
    "orp_mv", "horas_residencia", "bateria_pct",
]
df_raw.select(VARIABLES_NUMERICAS).describe().show()  # count/mean/stddev/min/max de cada variable numérica

+-------+------------------+------------------+------------------+------------------+------------------+-------------------+------------------+------------------+------------------+-------------------+
|summary|      distancia_cm|       temp_aire_c|      hum_aire_pct|            litros| presion_linea_kpa|         caudal_lpm|       temp_agua_c|            orp_mv|  horas_residencia|        bateria_pct|
+-------+------------------+------------------+------------------+------------------+------------------+-------------------+------------------+------------------+------------------+-------------------+
|  count|             34560|             34560|             34560|             34353|              4320|               4320|              4320|              4320|              4320|              34560|
|   mean| 32.08764154038251|12.999168981481482| 93.19020543981485|  791.569440805752|17.703981481481488|0.43023148148148166|12.976537037037028| 557.9788425925932|11.974513888888833|  99.922245

## 8. Correlación de cada variable candidata con `litros`

In [8]:
PREDICTORES_CANDIDATOS = [
    "distancia_cm", "temp_aire_c", "hum_aire_pct",
    "presion_linea_kpa", "caudal_lpm", "temp_agua_c",
    "orp_mv", "horas_residencia", "bateria_pct",
]
for c in PREDICTORES_CANDIDATOS:
    r = df_raw.stat.corr(c, "litros")     # correlación de Pearson; Spark ignora automáticamente los pares con null
    print(f"{c:22s} correlacion con litros: {r:+.4f}")

# caudal_lpm (+0.0015) y presion_linea_kpa (+0.0380) prácticamente no se mueven junto con litros:
# son candidatas a salir del modelo (Corrección 3, más abajo).

distancia_cm           correlacion con litros: -0.7466
temp_aire_c            correlacion con litros: +0.3492
hum_aire_pct           correlacion con litros: -0.3309
presion_linea_kpa      correlacion con litros: +0.0380
caudal_lpm             correlacion con litros: +0.0015
temp_agua_c            correlacion con litros: -0.1789
orp_mv                 correlacion con litros: -0.1384
horas_residencia       correlacion con litros: -0.3042
bateria_pct            correlacion con litros: +0.0750


## 9. Corrección 3 — Dos conjuntos de predictores en vez de uno

En el notebook original había un solo conjunto de 9 predictores, usado solo con `horeb_master_01`.
Aquí se separan en dos conjuntos con propósitos distintos:

- **`PREDICTORES_EXTENDIDOS`** (7 variables): las mismas de antes, **sin** `caudal_lpm` ni `presion_linea_kpa`
  (correlación e importancia casi nulas). Solo se pueden usar con `horeb_master_01`, porque son las únicas
  que ese dispositivo tiene.
- **`PREDICTORES_UNIVERSALES`** (6 variables): las que **sí** existen en los 8 dispositivos. Se agregan
  `capacidad_litros` y `altura_tanque_cm` — el notebook original nunca las probó como predictoras porque, al
  filtrar solo el master, esas columnas eran constantes (sin variación, sin utilidad). Pero **si el modelo
  va a servir para los 8 dispositivos**, esas dos columnas son las que le dicen al modelo el tamaño de cada
  tanque — sin ellas, una misma `distancia_cm` significa cantidades de agua distintas según el tanque.

In [9]:
# 7 predictores: sin caudal_lpm ni presion_linea_kpa (Correccion 3)
PREDICTORES_EXTENDIDOS = [
    "distancia_cm", "temp_aire_c", "hum_aire_pct",
    "temp_agua_c", "orp_mv", "horas_residencia", "bateria_pct",
]

# 6 predictores disponibles en los 8 dispositivos (incluye geometria del tanque)
PREDICTORES_UNIVERSALES = [
    "distancia_cm", "temp_aire_c", "hum_aire_pct", "bateria_pct",
    "capacidad_litros", "altura_tanque_cm",
]

print(f"Extendidos ({len(PREDICTORES_EXTENDIDOS)}): {PREDICTORES_EXTENDIDOS}")
print(f"Universales ({len(PREDICTORES_UNIVERSALES)}): {PREDICTORES_UNIVERSALES}")

Extendidos (7): ['distancia_cm', 'temp_aire_c', 'hum_aire_pct', 'temp_agua_c', 'orp_mv', 'horas_residencia', 'bateria_pct']
Universales (6): ['distancia_cm', 'temp_aire_c', 'hum_aire_pct', 'bateria_pct', 'capacidad_litros', 'altura_tanque_cm']


## 10. Limpieza — ahora sobre los 8 dispositivos

El notebook original filtraba a `horeb_master_01` *antes* de limpiar. Como ahora se va a construir también
el Modelo B (universal, con los 8 dispositivos), la limpieza básica se aplica primero a **todo** el dataset,
usando solo las columnas que todos comparten. La limpieza específica de los sensores extendidos se hace
más abajo, solo sobre el subconjunto del master.

In [10]:
# Filtro de calidad de datos aplicable a cualquier dispositivo:
#  - litros no puede ser negativo
#  - distancia_cm tiene que ser positiva (es una medición física de distancia)
# Despues se eliminan filas con nulos en litros o en cualquiera de los predictores universales
df_valido = (
    df_raw.filter(
        (col("litros") >= 0) &
        (col("distancia_cm") > 0)
    )
    .na.drop(subset=["litros"] + PREDICTORES_UNIVERSALES)
)

print(f"Antes: {df_raw.count():,}  Despues: {df_valido.count():,}")

Antes: 34,560  Despues: 34,353


## 11. Duplicados por dispositivo + timestamp

In [11]:
# Ahora la clave de unicidad es (device_id, timestamp), no solo timestamp,
# porque hay 8 dispositivos reportando en los mismos instantes de tiempo
total = df_valido.count()
sin_dup = df_valido.dropDuplicates(["device_id", "timestamp"]).count()
print(f"Total: {total:,}, sin duplicar por (device_id, timestamp): {sin_dup:,}")
assert total == sin_dup, "Hay una fila duplicada para el mismo dispositivo y el mismo timestamp"

df_valido = df_valido.cache()   # se cachea porque se va a reusar varias veces mas abajo (particionado, filtros de cada modelo)

Total: 34,353, sin duplicar por (device_id, timestamp): 34,353


## 12. Particionado a Parquet

Se particiona el dataset **completo** (los 8 dispositivos) por `zona` y `AnioMes`. A diferencia del
notebook original — donde solo se había particionado la data ya filtrada de `horeb_master_01` y por eso
solo aparecía la carpeta `zona=Zona A` — ahora deberían aparecer las 3 zonas reales del dataset
(`Zona A`, `Zona B`, `Zona C`).

In [12]:
from pyspark.sql.functions import date_format

# Se agrega una columna auxiliar "AnioMes" (ej. "2026-07") para poder particionar por mes
df_particionable = df_valido.withColumn(
    "AnioMes", date_format(col("timestamp"), "yyyy-MM")
)

(
    df_particionable
    .repartition(8)                              # reparte en 8 particiones antes de escribir (una por dispositivo, a modo de ejemplo)
    .write.format("parquet")                     # formato columnar, eficiente para lectura analitica
    .mode("overwrite")                           # sobreescribe si ya existia una corrida anterior
    .partitionBy("zona", "AnioMes")               # crea una carpeta por combinacion zona/mes
    .save(f"{ARTIFACTS}/tanques_particionado_v2")
)

carpetas = sorted(os.listdir(f"{ARTIFACTS}/tanques_particionado_v2"))
print("Carpetas de zona generadas:", [c for c in carpetas if c.startswith("zona=")])

Carpetas de zona generadas: ['zona=Zona A', 'zona=Zona B', 'zona=Zona C']


## 13. Verificación de partition pruning

In [14]:
df_verificacion = spark.read.parquet(f"{ARTIFACTS}/tanques_particionado_v2")
assert df_verificacion.count() == df_particionable.count()   # confirma que no se perdieron filas al escribir/leer
print(f"Verificado: {df_verificacion.count():,} filas")

# Al filtrar por una zona, el plan fisico debe mostrar PartitionFilters (Spark no escanea las otras zonas)
df_verificacion.filter(col("zona") == "Zona B").explain(True)
# Buscar en el "Physical Plan": PartitionFilters: [..., (zona = Zona B)]

Verificado: 34,353 filas
== Parsed Logical Plan ==
'Filter ('zona = Zona B)
+- Relation [device_id#4132,timestamp#4133,distancia_cm#4134,temp_aire_c#4135,hum_aire_pct#4136,nivel_cm_comp#4137,litros#4138,capacidad_litros#4139,altura_tanque_cm#4140,presion_linea_kpa#4141,caudal_lpm#4142,temp_agua_c#4143,orp_mv#4144,horas_residencia#4145,bateria_pct#4146,zona#4147,AnioMes#4148] parquet

== Analyzed Logical Plan ==
device_id: string, timestamp: timestamp, distancia_cm: double, temp_aire_c: double, hum_aire_pct: double, nivel_cm_comp: double, litros: double, capacidad_litros: double, altura_tanque_cm: double, presion_linea_kpa: double, caudal_lpm: double, temp_agua_c: double, orp_mv: double, horas_residencia: double, bateria_pct: double, zona: string, AnioMes: string
Filter (zona#4147 = Zona B)
+- Relation [device_id#4132,timestamp#4133,distancia_cm#4134,temp_aire_c#4135,hum_aire_pct#4136,nivel_cm_comp#4137,litros#4138,capacidad_litros#4139,altura_tanque_cm#4140,presion_linea_kpa#4141,cauda

## 14. Modelo A — extendido, solo `horeb_master_01`

Este modelo replica el propósito original del notebook: usar los 5 sensores extra que solo tiene el
dispositivo master, ahora sin `caudal_lpm` ni `presion_linea_kpa`.

In [15]:
# Filtra solo el dispositivo master
df_master = df_verificacion.filter(col("device_id") == "horeb_master_01")

# Limpieza especifica de los sensores extendidos:
#  - caudal_lpm y presion_linea_kpa no pueden ser negativos (aunque ya no se usen como predictores,
#    valores negativos indicarian una fila corrupta)
#  - se eliminan nulos en cualquiera de los 7 predictores extendidos o en litros
df_master_ext = (
    df_master
    .filter((col("caudal_lpm") >= 0) & (col("presion_linea_kpa") >= 0))
    .na.drop(subset=PREDICTORES_EXTENDIDOS + ["litros"])
)

print(f"horeb_master_01 -> antes: {df_master.count():,}  despues de limpiar: {df_master_ext.count():,}")

horeb_master_01 -> antes: 4,291  despues de limpiar: 4,291


## 15. Corrección 1 — Split temporal en vez de aleatorio

En vez de `randomSplit([0.8, 0.2])`, se ordena por tiempo y se corta en el percentil 80: el 80% de fechas
más antiguas es entrenamiento, el 20% más reciente es prueba. Así el modelo nunca "ve" datos del futuro
durante el entrenamiento, que es la condición real en la que se usaría en producción.

In [16]:
from pyspark.ml.feature import VectorAssembler
from pyspark.sql.functions import unix_timestamp

# VectorAssembler junta las columnas predictoras en un solo vector "features", como pide MLlib
ensamblador_a = VectorAssembler(inputCols=PREDICTORES_EXTENDIDOS, outputCol="features")
df_ml_a = ensamblador_a.transform(df_master_ext).select("features", "litros", "timestamp")

# Se convierte el timestamp a segundos (epoch) para poder calcular un percentil numerico sobre el tiempo
df_ml_a = df_ml_a.withColumn("ts_unix", unix_timestamp("timestamp"))

# approxQuantile calcula el percentil 80 de forma distribuida y aproximada (rapido en datasets grandes)
corte_a = df_ml_a.approxQuantile("ts_unix", [0.8], 0.001)[0]

df_train_a = df_ml_a.filter(col("ts_unix") <= corte_a)   # 80% de fechas mas antiguas -> entrenamiento
df_test_a  = df_ml_a.filter(col("ts_unix") >  corte_a)   # 20% de fechas mas recientes -> prueba

print(f"Modelo A - train: {df_train_a.count():,}  test: {df_test_a.count():,}")

Modelo A - train: 3,432  test: 859


## 16. Corrección 2 — `CrossValidator` para elegir el modelo

En vez de comparar configuraciones evaluando cada una una sola vez contra el mismo test set (lo que puede
llevar a elegir un modelo que solo le "atinó" a ese split en particular), se usa `CrossValidator`: parte
`df_train_a` en varios folds, prueba cada combinación de hiperparámetros en cada fold, y se queda con la
que tiene mejor RMSE promedio. El `df_test_a` (temporal) no se toca todavía.

In [17]:
from pyspark.ml.regression import LinearRegression, RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

evaluator_rmse = RegressionEvaluator(labelCol="litros", predictionCol="prediction", metricName="rmse")

# --- Regresion Lineal: grid de regularizacion (igual espiritu que la comparacion del notebook original,
#     pero ahora validada con folds en vez de "a ojo") ---
lr = LinearRegression(featuresCol="features", labelCol="litros")
grid_lr = (
    ParamGridBuilder()
    .addGrid(lr.regParam, [0.0, 0.1])          # sin regularizacion vs Ridge/ElasticNet leve
    .addGrid(lr.elasticNetParam, [0.0, 0.5])   # 0 = Ridge puro, 0.5 = mezcla L1+L2
    .build()
)
cv_lr = CrossValidator(
    estimator=lr, estimatorParamMaps=grid_lr,
    evaluator=evaluator_rmse, numFolds=3, seed=42, parallelism=2
)
cv_lr_model = cv_lr.fit(df_train_a)   # entrena y evalua las 4 combinaciones x 3 folds = 12 ajustes
print("A - LR RMSE promedio por combinacion (orden del grid):", [round(m, 4) for m in cv_lr_model.avgMetrics])

# --- Random Forest: grid de numero de arboles y profundidad ---
rf = RandomForestRegressor(featuresCol="features", labelCol="litros", seed=42)
grid_rf = (
    ParamGridBuilder()
    .addGrid(rf.numTrees, [30, 50])
    .addGrid(rf.maxDepth, [5, 8])
    .build()
)
cv_rf = CrossValidator(
    estimator=rf, estimatorParamMaps=grid_rf,
    evaluator=evaluator_rmse, numFolds=3, seed=42, parallelism=2
)
cv_rf_model = cv_rf.fit(df_train_a)
print("A - RF RMSE promedio por combinacion (orden del grid):", [round(m, 4) for m in cv_rf_model.avgMetrics])

A - LR RMSE promedio por combinacion (orden del grid): [39.039, 39.039, 39.0394, 39.035]
A - RF RMSE promedio por combinacion (orden del grid): [21.6365, 14.7702, 20.8541, 14.2393]


## 17. Evaluación final del Modelo A (una sola vez, contra el test temporal)

In [18]:
def evaluar(predicciones, nombre):
    """Calcula RMSE, R2 y MAE de un dataframe de predicciones y los imprime."""
    res = {}
    for m in ["rmse", "r2", "mae"]:
        ev = RegressionEvaluator(labelCol="litros", predictionCol="prediction", metricName=m)
        res[m.upper()] = ev.evaluate(predicciones)
    print(f"{nombre}: RMSE={res['RMSE']:.4f}  R2={res['R2']:.4f}  MAE={res['MAE']:.4f}")
    return res

# .bestModel es la mejor combinacion encontrada por CrossValidator; se evalua UNA sola vez sobre el test temporal
pred_lr_a = cv_lr_model.bestModel.transform(df_test_a)
pred_rf_a = cv_rf_model.bestModel.transform(df_test_a)

res_lr_a = evaluar(pred_lr_a, "Modelo A - LinearRegression (test temporal)")
res_rf_a = evaluar(pred_rf_a, "Modelo A - RandomForest   (test temporal)")

# Importancia de variables del Random Forest ganador
importancias_a = sorted(
    zip(PREDICTORES_EXTENDIDOS, cv_rf_model.bestModel.featureImportances.toArray()),
    key=lambda x: x[1], reverse=True
)
print()
for v, imp in importancias_a:
    print(f"{v:22s} {imp:.4f}")

Modelo A - LinearRegression (test temporal): RMSE=72.4590  R2=0.9163  MAE=16.2051
Modelo A - RandomForest   (test temporal): RMSE=16.8919  R2=0.9955  MAE=8.4541

horas_residencia       0.3853
distancia_cm           0.3235
orp_mv                 0.1976
temp_aire_c            0.0442
temp_agua_c            0.0194
hum_aire_pct           0.0187
bateria_pct            0.0113


## 18. Corrección 4 — Modelo B: universal, para los 8 dispositivos

Se repite el mismo proceso (VectorAssembler, split temporal, CrossValidator) pero con:
- `df_verificacion` completo (los 8 dispositivos, no solo el master)
- `PREDICTORES_UNIVERSALES` (incluye `capacidad_litros` y `altura_tanque_cm`, la geometría del tanque)

Este es el modelo que sí podría aplicarse a cualquier tanque nuevo que se agregue a la red, aunque no tenga
los sensores extendidos.

In [19]:
ensamblador_b = VectorAssembler(inputCols=PREDICTORES_UNIVERSALES, outputCol="features")
df_ml_b = ensamblador_b.transform(df_verificacion).select("features", "litros", "timestamp")
df_ml_b = df_ml_b.withColumn("ts_unix", unix_timestamp("timestamp"))

corte_b = df_ml_b.approxQuantile("ts_unix", [0.8], 0.001)[0]
df_train_b = df_ml_b.filter(col("ts_unix") <= corte_b)
df_test_b  = df_ml_b.filter(col("ts_unix") >  corte_b)
print(f"Modelo B - train: {df_train_b.count():,}  test: {df_test_b.count():,}")

# Mismos grids de hiperparametros que en el Modelo A, para comparar en igualdad de condiciones
cv_lr_b = CrossValidator(estimator=lr, estimatorParamMaps=grid_lr, evaluator=evaluator_rmse,
                          numFolds=3, seed=42, parallelism=2)
cv_lr_model_b = cv_lr_b.fit(df_train_b)

cv_rf_b = CrossValidator(estimator=rf, estimatorParamMaps=grid_rf, evaluator=evaluator_rmse,
                          numFolds=3, seed=42, parallelism=2)
cv_rf_model_b = cv_rf_b.fit(df_train_b)

pred_lr_b = cv_lr_model_b.bestModel.transform(df_test_b)
pred_rf_b = cv_rf_model_b.bestModel.transform(df_test_b)

res_lr_b = evaluar(pred_lr_b, "Modelo B - LinearRegression (test temporal)")
res_rf_b = evaluar(pred_rf_b, "Modelo B - RandomForest   (test temporal)")

importancias_b = sorted(
    zip(PREDICTORES_UNIVERSALES, cv_rf_model_b.bestModel.featureImportances.toArray()),
    key=lambda x: x[1], reverse=True
)
print()
for v, imp in importancias_b:
    print(f"{v:22s} {imp:.4f}")

Modelo B - train: 27,461  test: 6,892
Modelo B - LinearRegression (test temporal): RMSE=98.8153  R2=0.8707  MAE=22.6596
Modelo B - RandomForest   (test temporal): RMSE=61.0005  R2=0.9507  MAE=42.2011

distancia_cm           0.6938
capacidad_litros       0.0900
temp_aire_c            0.0852
altura_tanque_cm       0.0751
hum_aire_pct           0.0354
bateria_pct            0.0205


## 19. Comparación final y guardado de los modelos ganadores

In [20]:
import pandas as pd

comparacion_final = pd.DataFrame([
    {"Modelo": "A - LinearRegression", "Alcance": "solo master", **res_lr_a},
    {"Modelo": "A - RandomForest",     "Alcance": "solo master", **res_rf_a},
    {"Modelo": "B - LinearRegression", "Alcance": "8 dispositivos", **res_lr_b},
    {"Modelo": "B - RandomForest",     "Alcance": "8 dispositivos", **res_rf_b},
])[["Modelo", "Alcance", "RMSE", "R2", "MAE"]]

display(comparacion_final.sort_values("RMSE"))

# Se elige el ganador de cada alcance (A y B) SEGUN el RMSE real obtenido arriba,
# en vez de asumir de antemano que Random Forest gana (aunque en este dataset asi sea)
modelo_a_ganador = cv_rf_model.bestModel if res_rf_a["RMSE"] <= res_lr_a["RMSE"] else cv_lr_model.bestModel
modelo_b_ganador = cv_rf_model_b.bestModel if res_rf_b["RMSE"] <= res_lr_b["RMSE"] else cv_lr_model_b.bestModel
print(f"Ganador A: {'RandomForest' if modelo_a_ganador is cv_rf_model.bestModel else 'LinearRegression'}")
print(f"Ganador B: {'RandomForest' if modelo_b_ganador is cv_rf_model_b.bestModel else 'LinearRegression'}")

modelo_a_ganador.write().overwrite().save(f"{ARTIFACTS}/modelo_litros_a_master")
modelo_b_ganador.write().overwrite().save(f"{ARTIFACTS}/modelo_litros_b_universal")
print(f"Modelo A guardado en {ARTIFACTS}/modelo_litros_a_master")
print(f"Modelo B guardado en {ARTIFACTS}/modelo_litros_b_universal")

,Modelo,Alcance,RMSE,R2,MAE
1,A - RandomForest,solo master,16.891876,0.995454,8.454074
3,B - RandomForest,8 dispositivos,61.000549,0.950732,42.201096
0,A - LinearRegression,solo master,72.458984,0.916343,16.205118
2,B - LinearRegression,8 dispositivos,98.815274,0.870717,22.659639


Ganador A: RandomForest
Ganador B: RandomForest
Modelo A guardado en artifacts/modelo_litros_a_master
Modelo B guardado en artifacts/modelo_litros_b_universal


## 20. Conclusiones

- El split temporal muestra que el desempeño real (sobre todo de la regresión lineal) era más bajo de lo
  que sugería el split aleatorio del notebook original — el `RMSE` de Random Forest en el Modelo A sube de
  ~13 (split aleatorio) a un valor más alto con split temporal, que es la estimación honesta.
- `CrossValidator` confirma que Random Forest es consistentemente mejor que la regresión lineal en ambos
  folds y en el test temporal, no solo en un split afortunado.
- El Modelo B (universal) tiene métricas más débiles que el Modelo A: tiene menos variables disponibles,
  pero a cambio puede aplicarse a los 8 dispositivos en vez de a 1 solo. Agregar `capacidad_litros` y
  `altura_tanque_cm` fue clave para que este modelo funcionara — sin la geometría del tanque, una misma
  `distancia_cm` significa volúmenes de agua muy distintos entre dispositivos.
- `caudal_lpm` y `presion_linea_kpa` quedaron fuera sin pérdida de desempeño, como anticipaba su
  correlación casi nula.
